### **Project: Tool-Calling Agent**

#### ***Objective:***
##### *Build an agent that can understand a user's request and decide whether to answer directly or use an appropriate tool.*

#### **Tools:**
1. Multiplication Tool
2. DuckDuckGo Search
3. Word Counter

#### ***Goal:***
##### ***Understand the complete tool-calling flow from user query to tool execution and final response.***

In [18]:
### Load the environment Variable
from dotenv import load_dotenv
load_dotenv()

True

In [19]:
### LLM
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-120b"
)

llm.invoke("What is Langchain? in a single sentence").content

'LangChain is a framework that enables developers to build applications powered by large language models by chaining together prompts, data sources, and tool integrations for flexible, context‑aware AI workflows.'

In [20]:
from langchain.tools import tool
### Tool 1 Multiplication Calculator on Two Numbers
@tool
def multiplication(a:int,b:int):
    """This is Tool is used to do multiplication on Two Numbers"""
    return a*b

In [21]:
#### Tool 2 Word Counter

@tool
def word_counter(text:str):
    """This Tool is used for Count the Number Of Words in a Text"""
    words = text.split()
    total_words = len(words)
    return total_words

In [22]:
#### Tool 3 DuckDuckGOsearch for live data
from langchain_community.tools import DuckDuckGoSearchResults

@tool
def live_search(query:str):
    """This Tool is used to search the live information or current data using DUck Duck Go Search"""
    tool = DuckDuckGoSearchResults()

    return tool.run(query)

In [23]:
tools = [multiplication,word_counter,live_search] #Combine all tools

In [24]:
#Tool Binding
llm_with_tools = llm.bind_tools(tools,strict=True)

In [25]:
response = llm_with_tools.invoke("What is 25 multiplied by 10?")
response

AIMessage(content='25\u202f×\u202f10\u202f=\u202f250.', additional_kwargs={'reasoning_content': 'The user asks "What is 25 multiplied by 10?" Simple multiplication. Could compute directly: 250. No need for tool. Provide answer.'}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 200, 'total_tokens': 251, 'completion_time': 0.107236089, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.034584887, 'prompt_tokens_details': None, 'queue_time': 0.422648869, 'total_time': 0.141820976}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_017482bd7f', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0336f-00c8-7992-a395-cb938a88c5fd-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 200, 'output_tokens': 51, 'total_tokens': 251, 'output_token_details': {'reasoning': 32}})

In [26]:
query = """How many words are in "LangChain makes building LLM applications easier"?"""

In [27]:
response = llm_with_tools.invoke(query)
response

AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "How many words are in \'LangChain makes building LLM applications easier\'?" We can count words. Use word_counter tool.', 'tool_calls': [{'id': 'fc_d1fd7bff-2227-47c2-aed4-76a5fd8a2a9d', 'function': {'arguments': '{"text":"LangChain makes building LLM applications easier"}', 'name': 'word_counter'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 66, 'prompt_tokens': 206, 'total_tokens': 272, 'completion_time': 0.140540999, 'completion_tokens_details': {'reasoning_tokens': 32}, 'prompt_time': 0.00861053, 'prompt_tokens_details': None, 'queue_time': 0.317443888, 'total_time': 0.149151529}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_4200b3f836', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0336f-0617-72e3-aa55-1f299fe0497a-0', tool_calls=[{'name': 'word_counter', 'args': {'text': 'LangC

In [28]:
response.tool_calls[0]

{'name': 'word_counter',
 'args': {'text': 'LangChain makes building LLM applications easier'},
 'id': 'fc_d1fd7bff-2227-47c2-aed4-76a5fd8a2a9d',
 'type': 'tool_call'}

In [29]:
from langchain_core.messages import HumanMessage,AIMessage,ToolMessage

message = [HumanMessage(query)]
message.append(response)

tool_result = word_counter.invoke(response.tool_calls[0])
message.append(tool_result)

response = llm_with_tools.invoke(message)
response


AIMessage(content='The phrase “LangChain makes building LLM applications easier” contains **6 words**.', additional_kwargs={'reasoning_content': 'The user asks: "How many words are in \'LangChain makes building LLM applications easier\'?" The word count is 6. So answer: 6 words.'}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 240, 'total_tokens': 303, 'completion_time': 0.134923193, 'completion_tokens_details': {'reasoning_tokens': 36}, 'prompt_time': 0.010971012, 'prompt_tokens_details': None, 'queue_time': 0.370113432, 'total_time': 0.145894205}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_c868cf1eaa', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0336f-0913-7110-9554-cee1a420cdc9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 240, 'output_tokens': 63, 'total_tokens': 303, 'output_token_details': {'reasoning': 36}})

In [30]:
response.content

'The phrase “LangChain makes building LLM applications easier” contains **6 words**.'

In [31]:
### Create a dynamic tool registry

tool_map = {tool.name:tool for tool in tools}
tool_map

{'multiplication': StructuredTool(name='multiplication', description='This is Tool is used to do multiplication on Two Numbers', args_schema=<class 'langchain_core.utils.pydantic.multiplication'>, func=<function multiplication at 0x00000285DFF828E0>),
 'word_counter': StructuredTool(name='word_counter', description='This Tool is used for Count the Number Of Words in a Text', args_schema=<class 'langchain_core.utils.pydantic.word_counter'>, func=<function word_counter at 0x00000285DFF82840>),
 'live_search': StructuredTool(name='live_search', description='This Tool is used to search the live information or current data using DUck Duck Go Search', args_schema=<class 'langchain_core.utils.pydantic.live_search'>, func=<function live_search at 0x00000285DFF827A0>)}

In [32]:
query = "Search DuckDuckGo for the latest LangChain information."

In [33]:
### Dynamic Tool Calling Loop
from langchain_core.messages import HumanMessage,ToolMessage

def tool_loop_execution(query):
    messages = [HumanMessage(query)]


    while True:

        response = llm_with_tools.invoke(messages)

        messages.append(response)

        if not response.tool_calls:
            break

        for tool_call in response.tool_calls:
            tool_name = tool_call['name']
            tool_args = tool_call['args']
            tool_id = tool_call['id']
            print("Tool Name:",tool_name)
            tool = tool_map[tool_name]

            results = tool.invoke(tool_args)

            messages.append(ToolMessage(content = str(results),tool_call_id=tool_id))

    return response



In [34]:
test_queries = [
    # No tool
    "What is LangChain?",
    "What is the capital of India?",
    "Explain what an LLM is in one sentence.",

    # Multiplication
    "What is 25 multiplied by 16?",
    "Calculate 125 × 48.",
    "Multiply 37 and 29.",

    # Word Counter
    'How many words are in "LangChain makes building LLM applications easier"?',
    'Count the words in "Agents can use tools to perform actions."',
    'How many words are there in "RAG combines retrieval with generation for better answers"?',

    # DuckDuckGo Search
    "Search DuckDuckGo for the latest LangChain news.",
    "What are the latest developments in LangGraph? Search the web.",
    "Search for the current LangChain documentation.",
    "What is the latest version of Python? Search the web.",

    # Multiple tools
    "Calculate 25 × 16 and search DuckDuckGo for the latest LangChain news.",
    'Multiply 45 by 20 and count the words in "LangChain agents can use multiple tools."',
    'Count the words in "RAG improves factual grounding" and search for the latest RAG developments.'
]

for query in test_queries:
    print("Query:",query)
    response = tool_loop_execution(query).content
    print("Response:",response)

Query: What is LangChain?
Response: **LangChain** is an open‑source framework that makes it easier to build applications powered by large language models (LLMs). It provides a collection of modular components, utilities, and best‑practice patterns so developers can focus on the **“chain” of logic** that connects LLMs with other data sources, tools, and user interfaces.

### Core ideas

| Concept | What it is | Why it matters |
|---------|------------|----------------|
| **Chains** | A sequence of calls that can combine LLM prompts, data retrieval, post‑processing, and other actions. | Lets you compose complex behavior (e.g., “search‑then‑summarize”) from simple building blocks. |
| **Prompt Templates** | Reusable, parameterized prompt strings (with variables, few‑shot examples, etc.). | Guarantees consistent prompting and makes it easy to swap in different variables or models. |
| **Agents** | An LLM that decides **which tool** (search, calculator, API, database, etc.) to invoke next, 